# Module 3 — Data Transformations with Snowflake

Module 3 is where a Tasty Bytes analyst turns raw operational data into something the business can actually use. You'll join weather data, transform temperature units with UDFs, run the same query in Snowpark for Python, ship it as an incrementally maintained dynamic table, and set up a stream + stored procedure so downstream tasks (Module 5) have something to process.

## Learning objectives
- Combine POS and weather data to build harmonized analytical views.
- Author SQL UDFs and a Snowpark Python stored procedure.
- Query Snowflake data from Python with Snowpark.
- Materialize a query incrementally with a **Dynamic Table**.
- Capture change data with a **Stream** and process it in a stored procedure.

## Prerequisites
- You've completed **Module 2** and the `tasty_bytes` database is loaded (`raw_pos`, `raw_customer`, `harmonized`, `analytics` schemas).
- `frostbyte_weathersource` share installed.

## What you'll build
- `tasty_bytes.harmonized.daily_weather_v` and related weather views for Hamburg, Germany.
- Two UDFs for unit conversion (`fahrenheit_to_celsius`, `inch_to_millimeter`).
- A Snowpark view `windspeed_hamburg_snowpark`.
- A Dynamic Table `daily_sales_hamburg` (target lag: 1 minute).
- A Stream `order_header_stream` and a Snowpark Python sproc `process_order_headers_stream` that consumes it into `daily_sales_hamburg_t`.

## 1. Set your session context

In [ ]:
USE ROLE accountadmin;
USE WAREHOUSE compute_wh;
USE DATABASE tasty_bytes;

## 2. Explore Hamburg sales with SQL

We'll start with a plain SQL exploration: daily sales in Hamburg, Germany for February 2022. The CTE generates a full 28-day date dimension so days with no sales still show up (zero-filled).

In [ ]:
-- Query to explore sales in the city of Hamburg, Germany
WITH _feb_date_dim AS 
    (
    SELECT DATEADD(DAY, SEQ4(), '2022-02-01') AS date FROM TABLE(GENERATOR(ROWCOUNT => 28))
    )
SELECT
    fdd.date,
    ZEROIFNULL(SUM(o.price)) AS daily_sales
FROM _feb_date_dim fdd
LEFT JOIN analytics.orders_v o
    ON fdd.date = DATE(o.order_ts)
    AND o.country = 'Germany'
    AND o.primary_city = 'Hamburg'
WHERE fdd.date BETWEEN '2022-02-01' AND '2022-02-28'
GROUP BY fdd.date
ORDER BY fdd.date ASC;

## 3. Join weather data

Tasty Bytes uses the Snowflake Marketplace share **Weather Source** to correlate sales with weather. This view harmonizes the raw Weather Source history with Tasty Bytes' `country` table so it's easy to filter by city and country.

In [ ]:
-- Create view that adds weather data for cities where Tasty Bytes operates
CREATE OR REPLACE VIEW tasty_bytes.harmonized.daily_weather_v
COMMENT = 'Weather Source Daily History filtered to Tasty Bytes supported Cities'
    AS
SELECT
    hd.*,
    TO_VARCHAR(hd.date_valid_std, 'YYYY-MM') AS yyyy_mm,
    pc.city_name AS city,
    c.country AS country_desc
FROM FROSTBYTE_WEATHERSOURCE.onpoint_id.history_day hd
JOIN FROSTBYTE_WEATHERSOURCE.onpoint_id.postal_codes pc
    ON pc.postal_code = hd.postal_code
    AND pc.country = hd.country
JOIN TASTY_BYTES.raw_pos.country c
    ON c.iso_country = hd.country
    AND c.city = hd.city_name;

Now query the view for Hamburg-specific weather in February 2022:

In [ ]:
-- Query the view to explore daily temperatures in Hamburg, Germany for anomalies
SELECT
    dw.country_desc,
    dw.city_name,
    dw.date_valid_std,
    AVG(dw.avg_temperature_air_2m_f) AS avg_temperature_air_2m_f
FROM harmonized.daily_weather_v dw
WHERE 1=1
    AND dw.country_desc = 'Germany'
    AND dw.city_name = 'Hamburg'
    AND YEAR(date_valid_std) = '2022'
    AND MONTH(date_valid_std) = '2' -- February
GROUP BY dw.country_desc, dw.city_name, dw.date_valid_std
ORDER BY dw.date_valid_std DESC;

In [ ]:
-- Query the view to explore wind speeds in Hamburg, Germany for anomalies
SELECT
    dw.country_desc,
    dw.city_name,
    dw.date_valid_std,
    MAX(dw.max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM tasty_bytes.harmonized.daily_weather_v dw
WHERE 1=1
    AND dw.country_desc IN ('Germany')
    AND dw.city_name = 'Hamburg'
    AND YEAR(date_valid_std) = '2022'
    AND MONTH(date_valid_std) = '2' -- February
GROUP BY dw.country_desc, dw.city_name, dw.date_valid_std
ORDER BY dw.date_valid_std DESC;

Persist the windspeed query as a view so we can reuse it later:

In [ ]:
-- Create a view that tracks windspeed for Hamburg, Germany
CREATE OR REPLACE VIEW tasty_bytes.harmonized.windspeed_hamburg
    AS
SELECT
    dw.country_desc,
    dw.city_name,
    dw.date_valid_std,
    MAX(dw.max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM harmonized.daily_weather_v dw
WHERE 1=1
    AND dw.country_desc IN ('Germany')
    AND dw.city_name = 'Hamburg'
GROUP BY dw.country_desc, dw.city_name, dw.date_valid_std
ORDER BY dw.date_valid_std DESC;

## 4. Author UDFs for unit conversion

Weather Source data is imperial (Fahrenheit, inches). Global stakeholders want metric. Two simple SQL UDFs bridge the gap and keep the conversion logic in one place.

In [ ]:
CREATE OR REPLACE FUNCTION tasty_bytes.analytics.fahrenheit_to_celsius(temp_f NUMBER(35,4))
  RETURNS NUMBER(35,4)
  AS
  $$
    (temp_f - 32) * (5/9)
  $$
;

In [ ]:
CREATE OR REPLACE FUNCTION tasty_bytes.analytics.inch_to_millimeter(inch NUMBER(35,4))
  RETURNS NUMBER(35,4)
  AS
  $$
    inch * 25.4
  $$
;

Now apply the UDFs to build a Hamburg-specific weather+sales view and a broader per-city daily metrics view.

In [ ]:
-- Apply UDFs and confirm successful execution
CREATE OR REPLACE VIEW harmonized.weather_hamburg
AS
SELECT
    fd.date_valid_std AS date,
    fd.city_name,
    fd.country_desc,
    ZEROIFNULL(SUM(odv.price)) AS daily_sales,
    ROUND(AVG(fd.avg_temperature_air_2m_f),2) AS avg_temperature_fahrenheit,
    ROUND(AVG(analytics.fahrenheit_to_celsius(fd.avg_temperature_air_2m_f)),2) AS avg_temperature_celsius,
    ROUND(AVG(fd.tot_precipitation_in),2) AS avg_precipitation_inches,
    ROUND(AVG(analytics.inch_to_millimeter(fd.tot_precipitation_in)),2) AS avg_precipitation_millimeters,
    MAX(fd.max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM harmonized.daily_weather_v fd
LEFT JOIN harmonized.orders_v odv
    ON fd.date_valid_std = DATE(odv.order_ts)
    AND fd.city_name = odv.primary_city
    AND fd.country_desc = odv.country
WHERE 1=1
    AND fd.country_desc = 'Germany'
    AND fd.city = 'Hamburg'
    AND fd.yyyy_mm = '2022-02'
GROUP BY fd.date_valid_std, fd.city_name, fd.country_desc
ORDER BY fd.date_valid_std ASC;

In [ ]:
-- Expand tracking to all cities and deploy view with this new information
CREATE OR REPLACE VIEW analytics.daily_city_metrics_v
COMMENT = 'Daily Weather Metrics and Orders Data'
AS
SELECT
    fd.date_valid_std AS date,
    fd.city_name,
    fd.country_desc,
    ZEROIFNULL(SUM(odv.price)) AS daily_sales,
    ROUND(AVG(fd.avg_temperature_air_2m_f),2) AS avg_temperature_fahrenheit,
    ROUND(AVG(tasty_bytes.analytics.fahrenheit_to_celsius(fd.avg_temperature_air_2m_f)),2) AS avg_temperature_celsius,
    ROUND(AVG(fd.tot_precipitation_in),2) AS avg_precipitation_inches,
    ROUND(AVG(tasty_bytes.analytics.inch_to_millimeter(fd.tot_precipitation_in)),2) AS avg_precipitation_millimeters,
    MAX(fd.max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM tasty_bytes.harmonized.daily_weather_v fd
LEFT JOIN tasty_bytes.harmonized.orders_v odv
    ON fd.date_valid_std = DATE(odv.order_ts)
    AND fd.city_name = odv.primary_city
    AND fd.country_desc = odv.country
GROUP BY fd.date_valid_std, fd.city_name, fd.country_desc;

## 5. Same transformation, this time in Snowpark for Python

Snowpark lets you write these transformations in Python, using DataFrame semantics that will feel familiar to pandas/PySpark users, while pushing the work down to Snowflake compute. The next cell starts a session with `get_active_session()` — no credentials needed inside a Snowflake Notebook.

In [ ]:
from snowflake.snowpark.functions import col, max as sp_max, year, month
from snowflake.snowpark.context import get_active_session

session = get_active_session()

Load the weather view and filter to Hamburg, February 2022 — same slice we queried in SQL.

In [ ]:
daily_weather = session.table("tasty_bytes.harmonized.daily_weather_v")

filtered_weather = daily_weather.filter(
    (col("country_desc") == "Germany") &
    (col("city_name") == "Hamburg") &
    (year(col("date_valid_std")) == 2022) &
    (month(col("date_valid_std")) == 2)
)

Aggregate max wind speed by date and sort descending — same shape as the SQL query above, but composed as chained DataFrame operations.

In [ ]:
aggregated_weather = filtered_weather.groupBy(
    "country_desc", "city_name", "date_valid_std"
).agg(
    sp_max("max_wind_speed_100m_mph").alias("max_wind_speed_100m_mph")
)

sorted_weather = aggregated_weather.sort(col("date_valid_std").desc())

Snowflake Notebooks eagerly display Snowpark DataFrames as an interactive Streamlit table when they're the last expression in a cell:

In [ ]:
sorted_weather

Persist the Snowpark result as a view so downstream consumers can use it just like any other Snowflake view.

In [ ]:
sorted_weather.create_or_replace_view("tasty_bytes.harmonized.windspeed_hamburg_snowpark")

And because `streamlit` is imported by default, we can pivot straight from the DataFrame into an inline chart:

In [ ]:
import streamlit as st

st.subheader("Max wind speed in Hamburg — February 2022")
st.bar_chart(sorted_weather.to_pandas(), x="DATE_VALID_STD", y="MAX_WIND_SPEED_100M_MPH")

## 6. Ship daily Hamburg sales as a Dynamic Table

A **Dynamic Table** is a query result that Snowflake keeps continuously up to date on your behalf — no manual refresh, no external orchestrator. You specify the SELECT and a **target lag** (how fresh the data should be), and Snowflake figures out the incremental refresh strategy.

Below we materialize daily Hamburg sales with a 1-minute target lag. Adjust the lag to trade freshness against compute cost — `1 minute` is aggressive and great for demos; `1 hour` or `1 day` is more typical for production.

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE tasty_bytes.raw_pos.daily_sales_hamburg
WAREHOUSE = 'COMPUTE_WH'
TARGET_LAG = '1 minute'
AS
SELECT
    CAST(oh.ORDER_TS AS DATE) AS date,
    COALESCE(SUM(oh.ORDER_TOTAL), 0) AS total_sales
FROM
    tasty_bytes.raw_pos.order_header oh
JOIN
    tasty_bytes.raw_pos.location loc
ON
    oh.LOCATION_ID = loc.LOCATION_ID
WHERE
    loc.CITY = 'Hamburg'
    AND loc.COUNTRY = 'Germany'
GROUP BY
    CAST(oh.ORDER_TS AS DATE);

SELECT * FROM tasty_bytes.raw_pos.daily_sales_hamburg;

Insert a new Hamburg order. Within `TARGET_LAG` (1 minute), the dynamic table should pick up the change automatically — no manual refresh call.

In [ ]:
-- Insert dummy data for a sale occurring at a location in Hamburg
INSERT INTO tasty_bytes.raw_pos.order_header (
    ORDER_ID, 
    TRUCK_ID, 
    LOCATION_ID, 
    CUSTOMER_ID, 
    DISCOUNT_ID, 
    SHIFT_ID, 
    SHIFT_START_TIME, 
    SHIFT_END_TIME, 
    ORDER_CHANNEL, 
    ORDER_TS, 
    SERVED_TS, 
    ORDER_CURRENCY, 
    ORDER_AMOUNT, 
    ORDER_TAX_AMOUNT, 
    ORDER_DISCOUNT_AMOUNT, 
    ORDER_TOTAL
) VALUES (
    123456789, 101, 4493, null, null, 123456789,
    '08:00:00', '16:00:00', null,
    '2024-03-09 12:30:45', null, 'USD',
    12.00, null, null, 12.35
);

Wait ~1 minute, then re-query the dynamic table. The new row should appear without any explicit refresh command.

In [ ]:
SELECT * FROM tasty_bytes.raw_pos.daily_sales_hamburg ORDER BY date DESC;

## 7. Capture changes with a Stream, process them with a stored procedure

A **Stream** tracks change data (INSERT/UPDATE/DELETE) on a source table so downstream consumers can process only what's new. Standard streams like the one below capture INSERTs and UPDATEs but do not emit DELETEs (see the [streams reference](https://docs.snowflake.com/en/user-guide/streams-intro#types-of-streams)).

In [ ]:
USE SCHEMA raw_pos;

CREATE OR REPLACE STREAM tasty_bytes.raw_pos.order_header_stream ON TABLE tasty_bytes.raw_pos.order_header;

Insert an order at a different Hamburg location, then peek at the stream. You should see the new row plus a `METADATA$ACTION = 'INSERT'` column.

In [ ]:
INSERT INTO tasty_bytes.raw_pos.order_header (
    ORDER_ID, 
    TRUCK_ID, 
    LOCATION_ID, 
    CUSTOMER_ID, 
    DISCOUNT_ID, 
    SHIFT_ID, 
    SHIFT_START_TIME, 
    SHIFT_END_TIME, 
    ORDER_CHANNEL, 
    ORDER_TS, 
    SERVED_TS, 
    ORDER_CURRENCY, 
    ORDER_AMOUNT, 
    ORDER_TAX_AMOUNT, 
    ORDER_DISCOUNT_AMOUNT, 
    ORDER_TOTAL
) VALUES (
    123456789, 101, 1234, null, null, 123456789,
    '08:00:00', '16:00:00', null,
    '2023-07-01 12:30:45', null, 'USD',
    50.00, null, null, 52.50
);

SELECT * FROM tasty_bytes.raw_pos.order_header_stream;

Now delete that row. Because this is a **standard** stream, the DELETE won't produce a captured action — the stream is designed for append-oriented consumption.

In [ ]:
DELETE FROM tasty_bytes.raw_pos.order_header WHERE order_id = 123456789;

-- This won't return the deleted action in the stream because of how standard streams work
-- See: https://docs.snowflake.com/en/user-guide/streams-intro#types-of-streams
SELECT * FROM tasty_bytes.raw_pos.order_header_stream;

## 8. Author a Snowpark Python stored procedure to consume the stream

The stored procedure below reads the stream, filters to Hamburg-Germany orders, aggregates daily sales, and appends the results to `raw_pos.daily_sales_hamburg_t`. In Module 5 we'll schedule this sproc with a Task.

In [ ]:
-- Create the stored procedure, define its logic with Snowpark for Python, write sales to raw_pos.daily_sales_hamburg_t
CREATE OR REPLACE PROCEDURE tasty_bytes.raw_pos.process_order_headers_stream()
  RETURNS STRING
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.10'
  HANDLER ='process_order_headers_stream'
  PACKAGES = ('snowflake-snowpark-python')
AS
$$
import snowflake.snowpark.functions as F
from snowflake.snowpark import Session

def process_order_headers_stream(session: Session) -> float:
    # Query the stream
    recent_orders = session.table("order_header_stream").filter(F.col("METADATA$ACTION") == "INSERT")
    
    # Look up location of the orders in the stream using the LOCATIONS table
    locations = session.table("location")
    hamburg_orders = recent_orders.join(
        locations,
        recent_orders["LOCATION_ID"] == locations["LOCATION_ID"]
    ).filter(
        (locations["CITY"] == "Hamburg") &
        (locations["COUNTRY"] == "Germany")
    )
    
    # Calculate the sum of sales in Hamburg
    total_sales = hamburg_orders.group_by(F.date_trunc('DAY', F.col("ORDER_TS"))).agg(
        F.coalesce(F.sum("ORDER_TOTAL"), F.lit(0)).alias("total_sales")
    )
    
    # Select the columns with proper aliases and convert to date type
    daily_sales = total_sales.select(
        F.col("DATE_TRUNC('DAY', ORDER_TS)").cast("DATE").alias("DATE"),
        F.col("total_sales")
    )
    
    # Write the results to the DAILY_SALES_HAMBURG_T table
    total_sales.write.mode("append").save_as_table("raw_pos.daily_sales_hamburg_t")
    
    # Return a message indicating the operation was successful
    return "Daily sales for Hamburg, Germany have been successfully written to raw_pos.daily_sales_hamburg_t"
$$;

Test the sproc end-to-end: insert a Hamburg order, confirm it's captured, then call the sproc.

In [ ]:
-- Insert dummy data for a sale occurring at a location in Hamburg
INSERT INTO tasty_bytes.raw_pos.order_header (
    ORDER_ID, 
    TRUCK_ID, 
    LOCATION_ID, 
    CUSTOMER_ID, 
    DISCOUNT_ID, 
    SHIFT_ID, 
    SHIFT_START_TIME, 
    SHIFT_END_TIME, 
    ORDER_CHANNEL, 
    ORDER_TS, 
    SERVED_TS, 
    ORDER_CURRENCY, 
    ORDER_AMOUNT, 
    ORDER_TAX_AMOUNT, 
    ORDER_DISCOUNT_AMOUNT, 
    ORDER_TOTAL
) VALUES (
    123456789, 101, 4493, null, null, 123456789,
    '08:00:00', '16:00:00', null,
    '2023-07-01 12:30:45', null, 'USD',
    41.30, null, null, 45.80
);

-- Confirm the insert
SELECT * FROM tasty_bytes.raw_pos.order_header WHERE location_id = 4493;

In [ ]:
-- Call the stored procedure
CALL tasty_bytes.raw_pos.process_order_headers_stream();

-- Confirm the insert to the daily_sales_hamburg_t table
SELECT * FROM tasty_bytes.raw_pos.daily_sales_hamburg_t;

## Recap

You transformed raw Tasty Bytes data every way Snowflake supports it:
- **SQL views** for on-demand transformations.
- **SQL UDFs** for reusable inline logic.
- **Snowpark for Python** for DataFrame-style transformations that still run entirely in Snowflake.
- **Dynamic Tables** for declaratively-refreshed materializations.
- **Streams + stored procedures** for CDC-style incremental processing.

Module 4 delivers the Hamburg dataset to end users through Streamlit apps and a Native App. Module 5 automates the stream → sproc pipeline you just built.